# Preprocessing Encoding Data Abu Vulkanik
Notebook ini melakukan tiga transformasi utama:
1. `volcano_filter` -> One Hot Encoding
2. `timestamp` -> numerik bulan (1-12)
3. `alert_level` -> Label Encoding

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [2]:
# Load data
df = pd.read_csv('outlier_treatment_outputs/java-ash-hysplit_outlier_handled.csv')
df.head()

,id,timestamp,volcano_filter,alert_level,latitude,longitude,elevation,tinggi_letusan_m,kec_angin_km_jam,arah_angin_deg,amplitudo,duration,jarak_km,luas_km2,sudut_deg,radius_km
0,1,2024-06-11 07:34:00 UTC,Merapi,Yellow,-7.54194,110.44194,2968,1000.0,4.7,270.0,50.0,66.5,32.511505,614.390650,303.257335,18.304149
1,2,2024-01-24 08:56:00 UTC,Merapi,Orange,-7.54194,110.44194,2968,1000.0,8.0,135.0,51.0,168.0,27.761756,460.792988,37.901873,15.892819
2,3,2023-03-15 03:36:00 UTC,Merapi,Orange,-7.54194,110.44194,2968,1200.0,7.1,90.0,65.0,133.0,44.625714,614.214150,265.635506,23.934437
3,4,2023-03-13 22:50:00 UTC,Merapi,Orange,-7.54194,110.44194,2968,1250.0,3.2,135.0,70.0,160.0,22.998759,184.285530,288.108280,11.770659
4,5,2023-03-12 09:19:00 UTC,Merapi,Orange,-7.54194,110.44194,2968,1250.0,8.3,315.0,70.0,145.0,39.159595,307.107075,270.462663,20.809694


In [3]:
# Ubah timestamp menjadi angka bulan 1-12
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce').dt.month

# Label encoding untuk alert_level (deterministik berdasarkan urutan alfabet)
alert_mapping = {label: idx for idx, label in enumerate(sorted(df['alert_level'].dropna().unique()))}
df['alert_level'] = df['alert_level'].map(alert_mapping).astype('Int64')

# One-hot encoding untuk volcano_filter
volcano_ohe = pd.get_dummies(df['volcano_filter'], prefix='volcano_filter', dtype=int)
df_encoded = pd.concat([df.drop(columns=['volcano_filter']), volcano_ohe], axis=1)

df_encoded.head()

,id,timestamp,alert_level,latitude,longitude,elevation,tinggi_letusan_m,kec_angin_km_jam,arah_angin_deg,amplitudo,duration,jarak_km,luas_km2,sudut_deg,radius_km,volcano_filter_Bromo,volcano_filter_Merapi,volcano_filter_Raung,volcano_filter_Semeru,volcano_filter_Slamet
0,1,6,3,-7.54194,110.44194,2968,1000.0,4.7,270.0,50.0,66.5,32.511505,614.390650,303.257335,18.304149,0,1,0,0,0
1,2,1,1,-7.54194,110.44194,2968,1000.0,8.0,135.0,51.0,168.0,27.761756,460.792988,37.901873,15.892819,0,1,0,0,0
2,3,3,1,-7.54194,110.44194,2968,1200.0,7.1,90.0,65.0,133.0,44.625714,614.214150,265.635506,23.934437,0,1,0,0,0
3,4,3,1,-7.54194,110.44194,2968,1250.0,3.2,135.0,70.0,160.0,22.998759,184.285530,288.108280,11.770659,0,1,0,0,0
4,5,3,1,-7.54194,110.44194,2968,1250.0,8.3,315.0,70.0,145.0,39.159595,307.107075,270.462663,20.809694,0,1,0,0,0


In [4]:
# Tampilkan mapping label alert_level
print('Mapping alert_level -> label:')
print(alert_mapping)

Mapping alert_level -> label:
{'Green': 0, 'Orange': 1, 'Red': 2, 'Yellow': 3}


In [5]:
# Simpan hasil preprocessing
output_path = 'outlier_treatment_outputs/java-ash-hysplit_handled.csv'
df_encoded.to_csv(output_path, index=False)
print(f'Data hasil encoding disimpan ke: {output_path}')

Data hasil encoding disimpan ke: outlier_treatment_outputs/java-ash-hysplit_handled.csv
